In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [ ]:
load_dotenv(override=True)

open_ai_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HF_TOKEN")

login(hf_token, add_to_git_credential=True)

### Accessing Llama

Yesterday you should have received approval to use this model:

https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

You can either use that today, or it's faster if you get approval for this model too.

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Select this link to see if you need to request approval too. Pick the version of Llama that you want below by commenting out one of these! Or skip Llama altogether.

# Models

In [ ]:
# instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

# NOW Quantising the Model (i.e -> Making it less performative so that our chip can run it) by cutting its values.

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory

# quant_config = BitsAndBytesConfig(load_in_4bit=True,
#                                   bnb_4bit_use_double_quant=True,
#                                   bnb_4bit_compute_dtype=torch.bfloat16,
#                                   bnb_4bit_quant_type="nf4")

# This is for CUDA config not for my Mac chip here is the one for me

import torch

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages,return_tensors="pt").to("mps")
print(inputs)

In [ ]:
# The Model
model = AutoModelForCausalLM.from_pretrained(LLAMA,device_map = "auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6